# finrag with vLLM: Mistral 7B on a free GPU

Serves **Mistral 7B Instruct v0.3 (vLLM)** with vLLM (OpenAI-compatible API) from `unsloth/mistral-7b-instruct-v0.3` at revision
`60b99dcbc7f853b7cd18472a607e8d045b96f03e`, then runs finrag's held-out French evaluation through the `vllm` provider
(`FINRAG_MODEL=mistral-7b-vllm`), and measures throughput with 1 vs 32 concurrent requests.

Kaggle: Accelerator **GPU T4 x2**, Internet on. Corpus: the 45 sheets frozen in the repository.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True,
                          capture_output=True, text=True).stdout)
print("GPUs:", NGPU)

In [ ]:
# vLLM in its own environment (Python 3.12), pinned
!pip -q install uv
!uv venv -q -p 3.12 /tmp/vllm-env
!uv pip install -q --python /tmp/vllm-env vllm==0.33.0
!/tmp/vllm-env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# Start the server at the pinned revision; T4 has no bfloat16, so float16
import time
import urllib.request

log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["/tmp/vllm-env/bin/vllm", "serve", "unsloth/mistral-7b-instruct-v0.3",
     "--revision", "60b99dcbc7f853b7cd18472a607e8d045b96f03e",
     "--served-model-name", "mistral-7b", "--dtype", "half",
     "--tensor-parallel-size", str(NGPU), "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for i in range(240):  # up to 20 minutes for download + load
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print(f"vLLM ready after {i * 5} s")
        break
    except OSError:
        if server.poll() is not None:
            break
        time.sleep(5)
if server.poll() is not None:
    print(open("/kaggle/working/vllm.log").read()[-4000:])
    raise RuntimeError("vLLM did not start")
print(urllib.request.urlopen("http://localhost:8000/v1/models").read().decode()[:300])

In [ ]:
# finrag at the vllm-provider branch, with its locked dependencies
%cd /tmp
!rm -rf finrag && git clone -q --branch vllm-provider https://github.com/HadiJamalAhmad/finrag.git
%cd /tmp/finrag
!uv sync --frozen -q && git log --oneline -1

In [ ]:
import os

os.environ.update({
    "VLLM_URL": "http://localhost:8000/v1", "FINRAG_MODEL": "mistral-7b-vllm", "FINRAG_GPU": "T4",
    "FINRAG_DATA": "eval/fixtures/fr-interieur", "PYTHONUNBUFFERED": "1",
    "FINRAG_DOMAIN": "documents",
    "FINRAG_EMBED_MODEL": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "FINRAG_COLLECTION": "docs_fr", "FINRAG_MIN_SCORE": "0.61", "FINRAG_TOP_K": "3",
})
!uv run finrag ingest > /dev/null
!uv run finrag eval eval/suites/fr-interieur-heldout.jsonl \
    --out eval/reports/fr-heldout-mistral-7b-vllm.json

In [ ]:
# Throughput: 1 request at a time vs 32 at once (continuous batching)
import asyncio
import json

import httpx

PROMPT = ("Réponds en français en citant la source. Question : quelles pièces faut-il "
          "fournir pour une première demande de passeport ? " + "Contexte. " * 300)

async def one(c):
    r = await c.post("http://localhost:8000/v1/chat/completions", timeout=600, json={
        "model": "mistral-7b", "temperature": 0, "max_tokens": 256,
        "messages": [{"role": "user", "content": PROMPT}]})
    return r.json()["usage"]["completion_tokens"]

async def bench(n, conc):
    sem = asyncio.Semaphore(conc)
    async with httpx.AsyncClient() as c:
        async def lim():
            async with sem:
                return await one(c)
        t0 = time.perf_counter()
        toks = await asyncio.gather(*[lim() for _ in range(n)])
        dt = time.perf_counter() - t0
    return {"requests": n, "concurrency": conc, "seconds": round(dt, 2),
            "output_tokens": sum(toks), "tokens_per_s": round(sum(toks) / dt, 1)}

res = {"sequential": await bench(8, 1), "concurrent": await bench(32, 32), "gpus": NGPU}
res["speedup"] = round(res["concurrent"]["tokens_per_s"] / res["sequential"]["tokens_per_s"], 1)
print(json.dumps(res, indent=1))
json.dump(res, open("eval/reports/vllm-throughput.json", "w"), indent=1)

In [ ]:
# Only the reports go to the notebook output (not the environments)
!mkdir -p /kaggle/working/reports
!cp eval/reports/*.json /kaggle/working/reports/
!ls -la /kaggle/working/reports
server.terminate()